Read Bronze CSV from ADLS

Cast Data Types

Important because OpenSky CSV often stores numbers as strings.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze_path1 = "abfss://real@uralibootcamp.dfs.core.windows.net/bronze/lights"

bronze_df1 = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(bronze_path1)
)

display(bronze_df1)

In [0]:
bronze_df1.printSchema()

In [0]:
from pyspark.sql.functions import col

silver_df1 = (
    bronze_df1
    .withColumn("icao24", col("icao24").cast("string"))
    .withColumn("callsign", col("callsign").cast("string"))
    .withColumn("origin_country", col("origin_country").cast("string"))
    .withColumn("time_position", col("time_position").cast("timestamp"))
    .withColumn("last_contact", col("last_contact").cast("timestamp"))
    .withColumn("longitude", col("longitude").cast("double"))
    .withColumn("latitude", col("latitude").cast("double"))
    .withColumn("baro_altitude", col("baro_altitude").cast("double"))
    .withColumn("on_ground", col("on_ground").cast("boolean"))
    .withColumn("velocity", col("velocity").cast("double"))
    .withColumn("true_track", col("true_track").cast("double"))
    .withColumn("vertical_rate", col("vertical_rate").cast("double"))
    .withColumn("sensors", col("sensors").cast("string"))
    .withColumn("geo_altitude", col("geo_altitude").cast("double"))
    .withColumn("squawk", col("squawk").cast("string"))
    .withColumn("spi", col("spi").cast("boolean"))
    .withColumn("position_source", col("position_source").cast("integer"))
)

Data Quality Cleaning

Remove invalid locations:

In [0]:
silver_df1 = (
    silver_df1
    .filter(
        (col("latitude").between(-90,90)) &
        (col("longitude").between(-180,180))
    )
)

In [0]:
silver_df1 = (
    silver_df1
    .dropDuplicates(
        [
            "icao24",
            "time_position"
        ]
    )
)

Check NULL count for every column 

In [0]:
from pyspark.sql.functions import col, sum

null_check_df1 = silver_df1.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in bronze_df1.columns
    ]
)

display(null_check_df1)

In [0]:
silver_df1 = silver_df1.drop("sensors")

Add Business Columns
Flight Speed Conversion

OpenSky velocity is meters/sec.

Convert to km/hour:

In [0]:
silver_df1 = silver_df1.withColumn(
    "speed_kmh",
    round(col("velocity") * 3.6,2)
)

Speed Category

For Flight Speed Analytics:

In [0]:
silver_df1 = (
    silver_df1
    .withColumn(
        "speed_category",
        when(col("speed_kmh") < 200,"Low")
        .when(col("speed_kmh") < 700,"Normal")
        .otherwise("High")
    )
)

Flight Direction Category

For air corridor analysis:

In [0]:
from pyspark.sql.functions import col, when

silver_df1 = (
    silver_df1
    .withColumn(
        "direction",
        when(
            (col("true_track") >= 0) &
            (col("true_track") < 90),
            "North-East"
        )
        .when(
            (col("true_track") >= 90) &
            (col("true_track") < 180),
            "South-East"
        )
        .when(
            (col("true_track") >= 180) &
            (col("true_track") < 270),
            "South-West"
        )
        .when(
            (col("true_track") >= 270) &
            (col("true_track") <= 360),
            "North-West"
        )
        .otherwise("Unknown")
    )
)

Aircraft Safety Anomaly Flags
Detect aircraft moving on ground

In [0]:
silver_df1 = (
    silver_df1
    .withColumn(
        "ground_movement_flag",
        when(
            (col("on_ground")==True) &
            (col("velocity") > 20),
            1
        )
        .otherwise(0)
    )
)

Detect unusual speed

In [0]:
silver_df1 = (
    silver_df1
    .withColumn(
        "speed_anomaly_flag",
        when(
            col("speed_kmh") > 1200,
            1
        )
        .otherwise(0)
    )
)

Air Traffic Density Region

Create geographic zones.

In [0]:
silver_df1 = (
    silver_df1
    .withColumn(
        "region",
        concat(
            round(col("latitude"),0),
            lit("_"),
            round(col("longitude"),0)
        )
    )
)

Add Processing Metadata

In [0]:
silver_df1 = (
    silver_df1
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

silver_clean_df1 = (
    silver_df1
    .filter(col("icao24").isNotNull())
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())
    .withColumn("silver_timestamp", current_timestamp())
)

Write Silver Delta Table

Even if Bronze is CSV, Silver should be Delta.

In [0]:
silver_path1 = (
"abfss://real@uralibootcamp.dfs.core.windows.net/silver/flights"
)


(
silver_df1.write
.format("delta")
.mode("overwrite")
.option("mergeSchema","true")
.save(silver_path1)
)

Add Timestamp Columns (History Tracking)

Your Silver table should have:

In [0]:
from pyspark.sql.functions import *


silver_df1 = (
    silver_df1
    .withColumn(
        "created_timestamp",
        current_timestamp()
    )
    .withColumn(
        "updated_timestamp",
        current_timestamp()
    )
    .withColumn(
        "effective_start_date",
        current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)

In [0]:
(
silver_df1.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema","true")
.save(silver_path1)
)

In [0]:
silver_df1 .write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("openskyreal.silver.flights")

UPSERT using MERGE

Use case:

OpenSky sends the same aircraft again with updated:

location
speed
direction
altitude

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forPath(
    spark,
    silver_path1
)

(
silver_table.alias("target")
.merge(
    silver_df1.alias("source"),
    """
    target.icao24 = source.icao24
    AND target.time_position = source.time_position
    """
)
.whenMatchedUpdate(
    set={
        "latitude": "source.latitude",
        "longitude": "source.longitude",
        "velocity": "source.velocity",
        "speed_kmh": "source.speed_kmh",
        "silver_load_timestamp": "current_timestamp()"
    }
)
.whenNotMatchedInsertAll()
.execute()
)

In [0]:
%sql
DROP TABLE IF EXISTS openskyreal.silver.upsert;

In [0]:

silver_df1.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("openskyreal.silver.upsert")

SCD Type 1 (Current Data)

Use when you only need latest aircraft status.
Aircraft ABC123

Old:
speed=700

New:
speed=850


Replace old value

MERGE handles this.

Use:

Silver_Current_Flights

Table:

icao24
latitude
longitude
speed
country
silver_load_timestamp

SCD Type 2 (History Tracking)

Use when you want flight movement history.

we can analyze:

flight route
movement pattern
congestion
anomaly

In [0]:
from pyspark.sql.functions import current_timestamp, lit

silver_scd_df = (
    silver_df1
    .withColumn(
        "effective_start_date",
        current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)

Rewrite Delta table with SCD columns

Because your table was created without these columns:

In [0]:
(
silver_scd_df.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema","true")
.save(silver_path1)
)

In [0]:
from delta.tables import DeltaTable

silver_table1 = DeltaTable.forPath(
    spark,
    silver_path1
)

(
silver_table1.alias("target")
.merge(
    silver_scd_df.alias("source"),
    """
    target.icao24 = source.icao24
    AND target.is_current = true
    """
)
.whenMatchedUpdate(
    condition="""
    target.latitude <> source.latitude
    OR target.longitude <> source.longitude
    OR target.velocity <> source.velocity
    """,
    set={
        "effective_end_date": "current_timestamp()",
        "is_current": "false"
    }
)
.whenNotMatchedInsert(
    values={
        "icao24": "source.icao24",
        "latitude": "source.latitude",
        "longitude": "source.longitude",
        "velocity": "source.velocity",
        "effective_start_date": "current_timestamp()",
        "effective_end_date": "NULL",
        "is_current": "true"
    }
)
.execute()
)

In [0]:
%sql
DROP TABLE IF EXISTS opensky.silver.scd2_type;

In [0]:
silver_scd_df .write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("openskyreal.silver.scd2_type")

OPTIMIZE Silver Table

After many batch loads:

Purpose:

compact small files
improve query speed

In [0]:
%sql
OPTIMIZE openskyreal.silver.flights;

VACUUM

Remove old unused files:

VACUUM opensky.silver.flights
RETAIN 168 HOURS;

History Tracking

Delta automatically keeps versions.

Check history:

In [0]:
%sql
DESCRIBE HISTORY openskyreal.silver.flights;

In [0]:
%sql
DESCRIBE HISTORY opensky.silver.upsert;

In [0]:
%sql
DESCRIBE HISTORY openskyreal.silver.scd2_type;

Time Travel

You can query previous versions:

In [0]:
%sql
SELECT *
FROM openskyreal.silver.flights
VERSION AS OF 0;

In [0]:
%sql 
select * from openskyreal.silver.upsert;

In [0]:
%sql 
select * from openskyreal.silver.scd2_type;